In [6]:
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        if (candidate / ".git").exists():
            return candidate

    raise RuntimeError("MineSkynet repository root not found")


PROJECT_ROOT = find_repo_root(Path.cwd())

RAW_DATA = (
    PROJECT_ROOT
    / "MineMA-Model-Fine-Tuning"
    / ".cache"
    / "datasets"
    / "minecraft_qa"
    / "raw"
    / "minecraft_instruction_dataset.json"
)

FINETUNING_ROOT = (
    PROJECT_ROOT
    / "research"
    / "mineskynet_finetuning"
)

PROCESSED_DATA = FINETUNING_ROOT / "data" / "processed"
SPLIT_DATA = FINETUNING_ROOT / "data" / "splits"

print(PROJECT_ROOT)
print(RAW_DATA)
print(RAW_DATA.exists())

/home/pluto2479/Documents/MineSkynet
/home/pluto2479/Documents/MineSkynet/MineMA-Model-Fine-Tuning/.cache/datasets/minecraft_qa/raw/minecraft_instruction_dataset.json
True


In [ ]:
import json
from collections import Counter

path = RAW_DATA

print(f"Loading: {path}")
print(f"File size: {path.stat().st_size:,} bytes")

with path.open("r", encoding="utf-8") as f:
    rows = json.load(f)

print(f"\nTop-level type: {type(rows).__name__}")
print(f"Row count: {len(rows):,}")

row_types = Counter(type(row).__name__ for row in rows)
print(f"Row types: {dict(row_types)}")

all_keys = Counter()
for row in rows:
    if isinstance(row, dict):
        all_keys.update(row.keys())

print(f"Keys: {dict(all_keys)}")

for column in ("instruction", "input", "output"):
    values = [
        row.get(column)
        for row in rows
        if isinstance(row, dict)
    ]

    type_counts = Counter(type(value).__name__ for value in values)
    missing = sum(column not in row for row in rows if isinstance(row, dict))
    blank = sum(
        value is None or str(value).strip() == ""
        for value in values
    )

    print(f"\n[{column}]")
    print(f"Types: {dict(type_counts)}")
    print(f"Missing: {missing:,}")
    print(f"Blank/null: {blank:,}")

    abnormal = [
        (index, value)
        for index, row in enumerate(rows)
        if isinstance(row, dict)
        and (value := row.get(column)) is not None
        and not isinstance(value, str)
    ][:5]

    if abnormal:
        print("Non-string examples:")
        for index, value in abnormal:
            print(
                f"  row={index}, "
                f"type={type(value).__name__}, "
                f"value={value!r}"
            )

pairs = [
    (
        str(row.get("instruction", "")).strip(),
        str(row.get("output", "")).strip(),
    )
    for row in rows
    if isinstance(row, dict)
]

duplicate_count = len(pairs) - len(set(pairs))
print(f"\nDuplicate instruction/output pairs: {duplicate_count:,}")

Loading: /home/pluto2479/Documents/MineSkynet/MineMA-Model-Fine-Tuning/.cache/datasets/minecraft_qa/raw/minecraft_instruction_dataset.json
File size: 126,364,710 bytes

Top-level type: list
Row count: 390,317
Row types: {'dict': 390317}
Keys: {'instruction': 390317, 'input': 390317, 'output': 390317}

[instruction]
Types: {'str': 390266, 'float': 51}
Missing: 0
Blank/null: 0
Non-string examples:
  row=181, type=float, value=nan
  row=187, type=float, value=nan
  row=254, type=float, value=nan
  row=10138, type=float, value=nan
  row=10139, type=float, value=nan

[input]
Types: {'str': 390317}
Missing: 0
Blank/null: 390,317

[output]
Types: {'str': 390168, 'float': 149}
Missing: 0
Blank/null: 0
Non-string examples:
  row=419, type=float, value=nan
  row=3487, type=float, value=nan
  row=5156, type=float, value=nan
  row=7907, type=float, value=nan
  row=10137, type=float, value=nan

Duplicate instruction/output pairs: 9,118


In [ ]:
import pandas as pd

df = pd.DataFrame(rows)

req_cols = ['instruction', 'input', 'output']

# 행 열 기본 구조
print('shape : ', df.shape)
print('columns : ', df.columns.tolist())

# 필요 열 존재 확인
missing_cols = None
assert not missing_cols, f"Missing columns: {missing_cols}"

# 열별 품질 통계 리스트
col_stat = []

for col in req_cols : 
    series = df[col]

    n_missing = 
    n_non_str = 
    n_blank_str = 
    n_unique = 

    col_stat.append(
        {
            'column' : col,
            'missing' : n_missing,
            'non_str' : n_non_str,
            'blank_str' : n_blank_str,
            'unique' : n_unique
        }
    )    

summary_df = pd.DataFrame(col_stat)
summary_df

shape :  (390317, 3)
columns :  ['instruction', 'input', 'output']


In [10]:
import pandas as pd

df = pd.DataFrame(rows)

print("Raw rows:", f"{len(df):,}")
print("\nRaw dtypes:")
print(df.dtypes)

valid_mask = (
    df["instruction"].map(lambda value: isinstance(value, str))
    & df["output"].map(lambda value: isinstance(value, str))
)

valid_df = df.loc[valid_mask].copy()

valid_df["instruction"] = valid_df["instruction"].str.strip()
valid_df["input"] = valid_df["input"].fillna("").astype(str).str.strip()
valid_df["output"] = valid_df["output"].str.strip()

nonempty_mask = (
    valid_df["instruction"].ne("")
    & valid_df["output"].ne("")
)

valid_df = valid_df.loc[nonempty_mask].copy()

print("\nAfter invalid/empty removal:", f"{len(valid_df):,}")

exact_duplicates = valid_df.duplicated(
    subset=["instruction", "output"],
    keep="first",
)

print("Exact duplicate rows:", f"{exact_duplicates.sum():,}")

deduplicated_df = valid_df.loc[~exact_duplicates].copy()

print("After exact deduplication:", f"{len(deduplicated_df):,}")

answer_counts = (
    deduplicated_df
    .groupby("instruction")["output"]
    .nunique()
)

conflicting_questions = answer_counts[answer_counts > 1]

print(
    "Questions with conflicting outputs:",
    f"{len(conflicting_questions):,}",
)

Raw rows: 390,317

Raw dtypes:
instruction    object
input          object
output         object
dtype: object

After invalid/empty removal: 390,129
Exact duplicate rows: 9,053
After exact deduplication: 381,076
Questions with conflicting outputs: 19,644


In [11]:
conflict_examples = (
    deduplicated_df[
        deduplicated_df["instruction"].isin(
            conflicting_questions.index
        )
    ]
    .sort_values("instruction")
    .groupby("instruction")
    .head(3)
)

conflict_examples[
    ["instruction", "output"]
].head(20)

,instruction,output
8016,"""Bedrock!""\nMake a BUD switch next to a bed th...","To enhance the bedrock trap, consider surround..."
33299,"""Bedrock!""\nMake a BUD switch next to a bed th...",What type of switch should you make next to a ...
122986,$response_goes_here,**Where is it recommended to build a Miniature...
122985,$response_goes_here,$response_goes_here
77884,**Response**,4.
152156,**Response**,**How many Emeralds are needed for a Journeyma...
152159,**Response**,**How many trades until disabled for an Appren...
68402,**response**,4.
68406,**response**,8.
166254,-----|,---|
